# 🩺 Student Health Risk v2 — CatBoost-Anchored Ensemble + MLP + Pseudo-Labeling

**Playground Series S6E7** — predict `health_condition` ∈ {`at-risk`, `unhealthy`, `fit`},
scored by **balanced accuracy**. v1 of this pipeline scored **LB 0.94924** (CV 0.94944 — CV tracks LB
within ~2e-4 here). v2 targets the 0.952x leader cluster with four upgrades, each validated offline:

1. **CatBoost gets the compute.** On the v1 run CatBoost was both the *best* single model
   (tuned OOF 0.94920 vs 0.94854/0.94754 for LGBM/XGB) and ~35× faster on GPU (101s vs 3640s).
   v2 averages **3 CatBoost seeds** and re-balances the blend toward it.
2. **Pseudo-labeling.** Confident test-set predictions (~same generator as train) are added to each
   fold's training data for a second round — worth roughly +0.001 in offline simulation.
3. **A small neural corrector.** An embedding MLP joins the blend at a *capped* low weight:
   tree-heavy blends beat neural-heavy ones, but a modest NN share fixes a subset of rows the
   trees miss.
4. **Hard-rule masks.** The target is rule-generated: train has *zero* `fit` above BMI 26.00,
   zero `unhealthy` below BMI 19.85, no `at-risk` above BMI 30.83, etc. Bounds are re-derived from
   train at runtime and impossible classes are zeroed before the tuned argmax.

As in v1, the decisive step for this metric is **per-class multiplier tuning** on out-of-fold
predictions (`argmax(w ⊙ p)` instead of `argmax(p)`) — every candidate blend is masked, tuned, and
selected purely on OOF, so CV stays honest.


In [ ]:
import os, time, warnings, gc
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, log_loss
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression
from scipy.optimize import minimize

import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

class CFG:
    n_folds      = 5
    seed         = 42
    cat_seeds    = [42, 7, 2026]
    es_rounds    = 200
    pseudo_conf  = 0.90          # min blended max-prob to accept a pseudo-label
    nn_fracs     = [0.0, 0.05, 0.10, 0.15]   # capped NN share in the blend
    data_dirs    = ['/kaggle/input/competitions/playground-series-s6e7',
                    '/kaggle/input/playground-series-s6e7',
                    'data']

DATA_DIR = next(d for d in CFG.data_dirs if os.path.exists(os.path.join(d, 'train.csv')))
print('Using data dir:', DATA_DIR)


In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')
sub   = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

TARGET  = 'health_condition'
CLASSES = ['at-risk', 'fit', 'unhealthy']
CAT_COLS = ['diet_type', 'stress_level', 'sleep_quality',
            'physical_activity_level', 'smoking_alcohol', 'gender']
NUM_COLS = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
            'step_count', 'exercise_duration', 'water_intake']

y = train[TARGET].map({c: i for i, c in enumerate(CLASSES)}).values
print(train.shape, test.shape)


In [ ]:
def _probe(train_fn):
    try:
        train_fn(); return True
    except Exception as e:
        print('  GPU unavailable ->', str(e)[:80]); return False

Xp = np.random.rand(64, 4); yp = np.random.randint(0, 3, 64)
print('Probing GPU...')
LGB_GPU = _probe(lambda: lgb.LGBMClassifier(device='gpu', n_estimators=5, num_class=3,
                                            objective='multiclass', verbose=-1).fit(Xp, yp))
XGB_GPU = _probe(lambda: XGBClassifier(device='cuda', tree_method='hist', n_estimators=5).fit(Xp, yp))
CAT_GPU = _probe(lambda: CatBoostClassifier(task_type='GPU', iterations=5, verbose=0,
                                            allow_writing_files=False).fit(Xp, yp))
try:
    import torch
    TORCH_GPU = torch.cuda.is_available()
except Exception:
    torch = None; TORCH_GPU = False
print(f'LGB GPU={LGB_GPU}  XGB GPU={XGB_GPU}  CAT GPU={CAT_GPU}  TORCH GPU={TORCH_GPU}')


## Features + hard-rule mining

Same engineered features as v1 (missing indicators, ratios, `stress×activity` / `stress×sleep_quality`
crosses). New: mine the **hard class bounds** from train — with 690k rows, a class never appearing
beyond a numeric threshold is a property of the generator, not sampling luck. The bounds are used to
zero out impossible classes in predictions.


In [ ]:
def build_features(df):
    X = df[NUM_COLS + CAT_COLS].copy()
    X['n_missing'] = X[NUM_COLS + CAT_COLS].isna().sum(axis=1).astype(np.int8)
    for c in NUM_COLS + CAT_COLS:
        X[f'{c}_na'] = df[c].isna().astype(np.int8)
    X['cal_per_step']   = X['calorie_expenditure'] / (X['step_count'] + 1)
    X['cal_per_exmin']  = X['calorie_expenditure'] / (X['exercise_duration'] + 1)
    X['steps_x_ex']     = X['step_count'] * X['exercise_duration']
    X['hr_x_bmi']       = X['heart_rate'] * X['bmi']
    X['sleep_x_water']  = X['sleep_duration'] * X['water_intake']
    X['activity_score'] = X['step_count'] / 15000 + X['exercise_duration'] / 100
    X['bmi_cat']        = pd.cut(X['bmi'], [0, 18.5, 25, 30, 100], labels=False)
    X['stress_activity'] = df['stress_level'].astype(str) + '_' + df['physical_activity_level'].astype(str)
    X['stress_sleepq']   = df['stress_level'].astype(str) + '_' + df['sleep_quality'].astype(str)
    for c in CAT_COLS + ['stress_activity', 'stress_sleepq']:
        X[c] = X[c].astype('category')
    return X

X      = build_features(train)
X_test = build_features(test)
FEATURES     = list(X.columns)
CAT_FEATURES = [c for c in FEATURES if str(X[c].dtype) == 'category']
print(len(FEATURES), 'features |', len(CAT_FEATURES), 'categorical')


In [ ]:
# ---- hard bounds from train (never violated by any row of that class) ----
EPS = 0.005
B = {
    'fit_bmi_max':    train.loc[y == 1, 'bmi'].max()            + EPS,
    'unh_bmi_min':    train.loc[y == 2, 'bmi'].min()            - EPS,
    'atr_bmi_max':    train.loc[y == 0, 'bmi'].max()            + EPS,
    'fit_sleep_min':  train.loc[y == 1, 'sleep_duration'].min() - EPS,
    'unh_hr_max':     train.loc[y == 2, 'heart_rate'].max()     + EPS,
}
print(B)

def apply_masks(P, df):
    """Zero out classes that are impossible given the mined hard bounds."""
    P = P.copy()
    bmi, slp, hr = df['bmi'].values, df['sleep_duration'].values, df['heart_rate'].values
    P[(bmi > B['fit_bmi_max'])   & ~np.isnan(bmi), 1] = 0.0
    P[(bmi < B['unh_bmi_min'])   & ~np.isnan(bmi), 2] = 0.0
    P[(bmi > B['atr_bmi_max'])   & ~np.isnan(bmi), 0] = 0.0
    P[(slp < B['fit_sleep_min']) & ~np.isnan(slp), 1] = 0.0
    P[(hr  > B['unh_hr_max'])    & ~np.isnan(hr),  2] = 0.0
    return P

# sanity: masks must not contradict any training label
Pm = apply_masks(np.ones((len(X), 3)), train)
viol = (Pm[np.arange(len(y)), y] == 0).sum()
print('training rows whose true class a mask would forbid:', viol, '(must be 0)')


## Round-1 models

Shared 5-fold stratified CV. `run_cv` optionally appends pseudo-labeled rows to each fold's
*training* part only — validation rows stay pure train, so OOF scores remain honest in round 2.


In [ ]:
skf   = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
FOLDS = list(skf.split(X, y))
CB_CLASS_W = (len(y) / (3 * np.bincount(y, minlength=3))).tolist()

def run_cv(name, fit_predict, Xtr_all, Xte_all, extra=None):
    """extra = (X_extra, y_extra) pseudo rows appended to each fold's training part."""
    t0 = time.time()
    oof = np.zeros((len(Xtr_all), 3))
    pt  = np.zeros((len(Xte_all), 3))
    for fold, (ti, vi) in enumerate(FOLDS):
        X_tr, y_tr = Xtr_all.iloc[ti], y[ti]
        if extra is not None:
            X_tr = pd.concat([X_tr, extra[0]], ignore_index=True)
            y_tr = np.concatenate([y_tr, extra[1]])
        va_p, te_p = fit_predict(X_tr, y_tr, Xtr_all.iloc[vi], y[vi], Xte_all)
        oof[vi] = va_p
        pt     += te_p / CFG.n_folds
        print(f'  fold {fold}: bal_acc={balanced_accuracy_score(y[vi], va_p.argmax(1)):.5f}', flush=True)
    print(f'{name}: OOF bal_acc={balanced_accuracy_score(y, oof.argmax(1)):.5f} '
          f'[{time.time()-t0:.0f}s]', flush=True)
    return oof, pt


In [ ]:
def fit_lgb(X_tr, y_tr, X_va, y_va, X_te):
    params = dict(objective='multiclass', num_class=3,
                  n_estimators=5000, learning_rate=0.045,
                  num_leaves=96, min_child_samples=40,
                  colsample_bytree=0.8, subsample=0.9, subsample_freq=1,
                  reg_alpha=1.0, reg_lambda=5.0, class_weight='balanced',
                  max_bin=127, random_state=CFG.seed, n_jobs=-1, verbose=-1)
    if LGB_GPU:
        params.update(device='gpu', gpu_use_dp=False)
    m = lgb.LGBMClassifier(**params)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(CFG.es_rounds, verbose=False)])
    return m.predict_proba(X_va), m.predict_proba(X_te)

def fit_xgb(X_tr, y_tr, X_va, y_va, X_te):
    m = XGBClassifier(objective='multi:softprob', num_class=3,
                      n_estimators=5000, learning_rate=0.04,
                      max_depth=8, min_child_weight=20,
                      colsample_bytree=0.8, subsample=0.9,
                      reg_alpha=1.0, reg_lambda=5.0,
                      tree_method='hist', enable_categorical=True,
                      device='cuda' if XGB_GPU else 'cpu',
                      early_stopping_rounds=CFG.es_rounds, eval_metric='mlogloss',
                      random_state=CFG.seed, n_jobs=-1, verbosity=0)
    sw = compute_sample_weight('balanced', y_tr)
    m.fit(X_tr, y_tr, sample_weight=sw, eval_set=[(X_va, y_va)], verbose=False)
    return m.predict_proba(X_va), m.predict_proba(X_te)

def fit_cat_multiseed(X_tr, y_tr, X_va, y_va, X_te):
    va_p = np.zeros((len(X_va), 3)); te_p = np.zeros((len(X_te), 3))
    for sd in CFG.cat_seeds:
        params = dict(loss_function='MultiClass', iterations=6000, learning_rate=0.06,
                      depth=8, l2_leaf_reg=5.0, random_strength=1.0,
                      class_weights=CB_CLASS_W, cat_features=CAT_FEATURES,
                      od_type='Iter', od_wait=CFG.es_rounds,
                      random_seed=sd, verbose=0, allow_writing_files=False)
        if CAT_GPU:
            params.update(task_type='GPU', devices='0')
        m = CatBoostClassifier(**params)
        m.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        va_p += m.predict_proba(X_va) / len(CFG.cat_seeds)
        te_p += m.predict_proba(X_te) / len(CFG.cat_seeds)
    return va_p, te_p

# CatBoost view: categorical NaNs -> explicit level
def cb_view(df):
    Xc = df.copy()
    for c in CAT_FEATURES:
        Xc[c] = Xc[c].cat.add_categories('missing').fillna('missing').astype(str)
    return Xc

Xc, Xc_test = cb_view(X), cb_view(X_test)


In [ ]:
oof_cat, pt_cat = run_cv('CatBoost x3', fit_cat_multiseed, Xc, Xc_test)
oof_xgb, pt_xgb = run_cv('XGBoost',     fit_xgb,           X,  X_test)
oof_lgb, pt_lgb = run_cv('LightGBM',    fit_lgb,           X,  X_test)
gc.collect()


## Neural corrector — embedding MLP

Small 2×256 MLP with categorical embeddings, class-weighted cross-entropy, cosine LR, best-epoch
snapshot per fold. Joins the blend at a **capped** weight — offline testing (and the general
experience in this competition) shows tree-heavy blends win, with the NN contributing corrections
for a limited subset of rows.


In [ ]:
if torch is not None:
    import torch.nn as nn
    torch.manual_seed(CFG.seed)

    def prep_nn(df, stats=None):
        num = df[NUM_COLS].copy()
        na  = num.isna().astype(np.float32).values
        if stats is None:
            stats = (num.mean(), num.std().replace(0, 1))
        num = ((num - stats[0]) / stats[1]).fillna(0).astype(np.float32).values
        ext = np.column_stack([
            (df['calorie_expenditure'] / (df['step_count'] + 1)).fillna(0),
            (df['step_count'] * df['exercise_duration']).fillna(0) / 1e6,
            (df['step_count'] / 15000 + df['exercise_duration'] / 100).fillna(0),
        ]).astype(np.float32)
        LEV = {'diet_type': ['balanced','non-veg','veg'], 'stress_level': ['high','low','medium'],
               'sleep_quality': ['average','good','poor'],
               'physical_activity_level': ['active','moderate','sedentary'],
               'smoking_alcohol': ['no','occasional','yes'], 'gender': ['female','male','other']}
        cats = np.column_stack([df[c].map({v: i for i, v in enumerate(LEV[c])}) for c in CAT_COLS])
        cats = pd.DataFrame(cats).fillna(3).astype(np.int64).values   # 3 = missing bucket
        return np.hstack([num, na, ext]), cats, stats

    Xn, Xcat, nn_stats = prep_nn(train)
    Xn_te, Xcat_te, _  = prep_nn(test, nn_stats)

    class MLP(nn.Module):
        def __init__(self, n_num, n_cat, card=4, emb=6):
            super().__init__()
            self.embs = nn.ModuleList([nn.Embedding(card, emb) for _ in range(n_cat)])
            d = n_num + n_cat * emb
            self.net = nn.Sequential(
                nn.Linear(d, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.25),
                nn.Linear(256, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.25),
                nn.Linear(256, 3))
        def forward(self, xn, xc):
            e = [m(xc[:, i]) for i, m in enumerate(self.embs)]
            return self.net(torch.cat([xn] + e, 1))

    dev  = 'cuda' if TORCH_GPU else 'cpu'
    cw_t = torch.tensor(len(y) / (3 * np.bincount(y, minlength=3)), dtype=torch.float32, device=dev)
    BS, EPOCHS = 4096, 14

    def nn_predict(model, xn, xc):
        model.eval(); out = []
        with torch.no_grad():
            for b in range(0, len(xn), 16384):
                out.append(torch.softmax(model(
                    torch.tensor(xn[b:b+16384]).to(dev),
                    torch.tensor(xc[b:b+16384]).to(dev)), 1).cpu().numpy())
        return np.vstack(out)

    t0 = time.time()
    oof_nn = np.zeros((len(y), 3)); pt_nn = np.zeros((len(test), 3))
    for fold, (ti, vi) in enumerate(FOLDS):
        model = MLP(Xn.shape[1], Xcat.shape[1]).to(dev)
        opt   = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
        lossf = nn.CrossEntropyLoss(weight=cw_t)
        xn_t, xc_t = torch.tensor(Xn[ti]), torch.tensor(Xcat[ti])
        y_t = torch.tensor(y[ti]); n = len(ti)
        best_vl, best_state = 1e9, None
        for ep in range(EPOCHS):
            model.train()
            perm = torch.randperm(n)
            for b in range(0, n, BS):
                j = perm[b:b+BS]
                opt.zero_grad()
                loss = lossf(model(xn_t[j].to(dev), xc_t[j].to(dev)), y_t[j].to(dev))
                loss.backward(); opt.step()
            sched.step()
            va_p = nn_predict(model, Xn[vi], Xcat[vi])
            vl = log_loss(y[vi], np.clip(va_p, 1e-9, 1),
                          sample_weight=compute_sample_weight('balanced', y[vi]))
            if vl < best_vl:
                best_vl = vl
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
        model.load_state_dict(best_state)
        oof_nn[vi] = nn_predict(model, Xn[vi], Xcat[vi])
        pt_nn     += nn_predict(model, Xn_te, Xcat_te) / CFG.n_folds
        print(f'  fold {fold}: bal_acc={balanced_accuracy_score(y[vi], oof_nn[vi].argmax(1)):.5f} '
              f'vloss={best_vl:.5f}', flush=True)
    print(f'MLP: OOF bal_acc={balanced_accuracy_score(y, oof_nn.argmax(1)):.5f} [{time.time()-t0:.0f}s]')
    HAS_NN = True
else:
    HAS_NN = False
    print('torch unavailable -> skipping NN')


## Mask → tune → select machinery

Every candidate blend is masked with the mined hard rules, then its per-class multipliers are
grid-searched (log-spaced coarse grid → local refinement) on OOF balanced accuracy.


In [ ]:
CLASS_COUNTS = np.bincount(y, minlength=3)

def bal_acc_fast(y_true, pred, counts):
    hits = np.zeros(3)
    np.add.at(hits, y_true[pred == y_true], 1)
    return (hits / counts).mean()

def tune_class_weights(P, y_true, lo=0.15, hi=30.0, n=40, refine=3):
    counts = np.bincount(y_true, minlength=3)
    best = (1.0, 1.0)
    best_s = bal_acc_fast(y_true, P.argmax(1), counts)
    g1 = g2 = np.logspace(np.log10(lo), np.log10(hi), n)
    for _ in range(refine + 1):
        for w1 in g1:
            for w2 in g2:
                s = bal_acc_fast(y_true, (P * np.array([1.0, w1, w2])).argmax(1), counts)
                if s > best_s:
                    best_s, best = s, (w1, w2)
        c1, c2 = best
        r1 = (g1[1] / g1[0]) ** 2; r2 = (g2[1] / g2[0]) ** 2
        g1 = np.logspace(np.log10(c1 / r1), np.log10(c1 * r1), 15)
        g2 = np.logspace(np.log10(c2 / r2), np.log10(c2 * r2), 15)
    return np.array([1.0, *best]), best_s

TREE_WEIGHTS = {          # (lgb, xgb, cat)
    'equal':      (1, 1, 1),
    'cat2':       (1, 1, 2),
    'cat3':       (1, 1, 3),
    'cat_xgb':    (0, 1, 2),
    'cat_lgb':    (1, 0, 2),
    'cat_only':   (0, 0, 1),
}

def select_blend(oofs, pts, tag):
    """oofs/pts: dicts with keys lgb/xgb/cat (+nn). Returns best (name, P_test, class_w, score)."""
    results = {}
    nn_fracs = CFG.nn_fracs if HAS_NN else [0.0]
    for wname, tw in TREE_WEIGHTS.items():
        tw = np.array(tw, dtype=float); tw /= tw.sum()
        P_tree  = tw[0] * oofs['lgb'] + tw[1] * oofs['xgb'] + tw[2] * oofs['cat']
        Pt_tree = tw[0] * pts['lgb']  + tw[1] * pts['xgb']  + tw[2] * pts['cat']
        for f in nn_fracs:
            P  = (1 - f) * P_tree  + f * (oofs['nn'] if HAS_NN else 0)
            Pt = (1 - f) * Pt_tree + f * (pts['nn']  if HAS_NN else 0)
            cw, s = tune_class_weights(apply_masks(P, train), y)
            results[f'{wname}+nn{f:.2f}'] = (Pt, cw, s)
    for name, (_, cw, s) in sorted(results.items(), key=lambda kv: -kv[1][2]):
        print(f'{tag} {name:18s} tuned={s:.5f}  class_w={np.round(cw, 3)}')
    best = max(results, key=lambda k: results[k][2])
    print(f'{tag} selected: {best}  (OOF {results[best][2]:.5f})')
    return best, *results[best]


In [ ]:
oofs1 = {'lgb': oof_lgb, 'xgb': oof_xgb, 'cat': oof_cat}
pts1  = {'lgb': pt_lgb,  'xgb': pt_xgb,  'cat': pt_cat}
if HAS_NN:
    oofs1['nn'] = oof_nn; pts1['nn'] = pt_nn

best1, PT1, CW1, S1 = select_blend(oofs1, pts1, 'R1')


## Pseudo-labeling round

Label the test set with the round-1 blend (masked + tuned), keep rows whose *raw* blended
confidence is high, and retrain the tree models with those rows appended to every fold's training
part. Validation stays pure-train, so round-2 OOF scores remain comparable and honest.


In [ ]:
PT1_masked = apply_masks(PT1, test)
pseudo_lab  = (PT1_masked * CW1).argmax(1)
pseudo_conf = PT1_masked.max(1)
keep = pseudo_conf >= CFG.pseudo_conf
print(f'pseudo rows: {keep.sum()} / {len(test)} ({keep.mean():.1%})  '
      f'label dist: {np.bincount(pseudo_lab[keep], minlength=3)}')

X_ps,  y_ps  = X_test[keep].reset_index(drop=True),  pseudo_lab[keep]
Xc_ps        = Xc_test[keep].reset_index(drop=True)


In [ ]:
oof_cat2, pt_cat2 = run_cv('CatBoost x3 +pseudo', fit_cat_multiseed, Xc, Xc_test, extra=(Xc_ps, y_ps))
oof_xgb2, pt_xgb2 = run_cv('XGBoost +pseudo',     fit_xgb,           X,  X_test,  extra=(X_ps,  y_ps))
oof_lgb2, pt_lgb2 = run_cv('LightGBM +pseudo',    fit_lgb,           X,  X_test,  extra=(X_ps,  y_ps))
gc.collect()


In [ ]:
oofs2 = {'lgb': oof_lgb2, 'xgb': oof_xgb2, 'cat': oof_cat2}
pts2  = {'lgb': pt_lgb2,  'xgb': pt_xgb2,  'cat': pt_cat2}
if HAS_NN:
    oofs2['nn'] = oof_nn; pts2['nn'] = pt_nn      # NN not retrained; still a valid blend component

best2, PT2, CW2, S2 = select_blend(oofs2, pts2, 'R2')

if S2 >= S1:
    PT_FINAL, CW_FINAL, S_FINAL, round_used = PT2, CW2, S2, 'round2(pseudo)'
else:
    PT_FINAL, CW_FINAL, S_FINAL, round_used = PT1, CW1, S1, 'round1'
print(f'FINAL: {round_used}  CV balanced accuracy = {S_FINAL:.5f}')


In [ ]:
P_sub = apply_masks(PT_FINAL, test) * CW_FINAL
pred  = P_sub.argmax(1)

sub[TARGET] = [CLASSES[i] for i in pred]
sub.to_csv('submission.csv', index=False)
print(sub[TARGET].value_counts(normalize=True).round(4))
sub.head()


### Notes
- All selection (blend weights, NN share, multipliers, round-1 vs round-2) happens on out-of-fold
  predictions of *real* train rows only — the printed CV number has tracked LB within ~2e-4.
- The hard-rule masks are re-derived from train at runtime; the sanity cell asserts they never
  contradict a training label.
- Runtime on T4 x2 ≈ 2.5–3 h (CatBoost seeds are ~2 min each on GPU; LightGBM dominates the budget).
